# HW4: The air-quality alert service

**Name:** _your name_ &nbsp;&nbsp; **Student ID:** _your id_

See `README.md` for the scenario, requirements and grading. `feed.py` (do not modify) gives you the history and the 2014 production feed. Put your features, model, service and drift detector in `hw4_submission.py`.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "2"   # keep OpenMP threads modest on shared machines

import json

import mlflow
import numpy as np
import pandas as pd

import feed
import hw4_submission as sub

SEED = 0
EXPERIMENT = "hw4-air-quality"
MODEL_NAME = "pm25-forecaster"
mlflow.set_tracking_uri("sqlite:///mlflow.db")   # browse: uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
mlflow.set_experiment(EXPERIMENT)
client = mlflow.MlflowClient()

/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
hourly = feed.load_hourly()                  # 2010-2014, one row per hour
print(hourly.shape, hourly.index.min(), hourly.index.max())
hourly.head()

(43824, 8) 2010-01-01 00:00:00 2014-12-31 23:00:00


,pm2.5,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir
time,,,,,,,,
2010-01-01 00:00:00,NaN,-21.0,-11.0,1021.0,NW,1.79,0.0,0.0
2010-01-01 01:00:00,NaN,-21.0,-12.0,1020.0,NW,4.92,0.0,0.0
2010-01-01 02:00:00,NaN,-21.0,-11.0,1019.0,NW,6.71,0.0,0.0
2010-01-01 03:00:00,NaN,-21.0,-14.0,1019.0,NW,9.84,0.0,0.0
2010-01-01 04:00:00,NaN,-20.0,-12.0,1018.0,NW,12.97,0.0,0.0


In [3]:
batches = feed.production_batches(hourly)    # 2014 as received in production, one frame per week
print(len(batches), "weekly batches; first:", batches[0].index[0], "last:", batches[-1].index[-1])

53 weekly batches; first: 2014-01-01 00:00:00 last: 2014-12-31 23:00:00


## 1. Features, baselines and the champion (history up to 2013)

In [4]:
# TODO

## 2. Register and serve

In [5]:
# TODO

## 3. Monitor the 2014 feed

In [6]:
# TODO

## 4. Challenger, comparison and promotion

In [7]:
# TODO

## 5. Check the submission module

In [8]:
for name in ["make_features", "build_pipeline", "create_app", "detect_drift"]:
    print(name, "defined" if callable(getattr(sub, name, None)) else "MISSING")

make_features defined
build_pipeline defined
create_app defined
detect_drift defined


## 6. Answers and export

A reported number is `{"value": float, "run_id": "<MLflow run id>", "metric": "<metric name in that run>"}`.

In [9]:
answers = {
    "student_id": None,                    # str
    "horizon_hours": None,                 # int
    "weather_policy": None,                # which weather your served model uses: "observed_at_origin_only" |
                                           # "observed_at_target_time" | "not_used"
    "oracle_gap_mae": None,                # float: validation MAE of your model minus that of the same model given
                                           #   weather observed at the target time
    "baseline_val_mae": None,              # reported number: best simple baseline, validation period
    "champion_val_mae": None,              # reported number: champion, same validation period
    "drifted_features": None,              # list of input columns you found drifting or faulty in 2014
    "fault_start": None,                   # "YYYY-MM-DD": when the problem you found started
    "champion_version": None,              # int: registry version first given the "champion" alias
    "challenger_version": None,            # int: registry version of your challenger
    "promoted": None,                      # bool: does "champion" point to the challenger at the end?
    "champion_mae_fresh": None,            # reported number: champion on your comparison window
    "challenger_mae_fresh": None,          # reported number: challenger on the same window
    "challenger_minus_champion_ci": None,  # [low, high]: 95% interval for the MAE difference
    "memo": {
        "decision": None,                  # "promote_challenger" | "keep_champion" | "rollback"
        "main_risk": None,                 # "sensor_fault" | "seasonal_change" | "weather_forecast_error" |
                                           # "data_gaps" | "none"
        "action_on_fault": None,           # "fix_input_and_retrain" | "fix_input_only" | "retrain_on_faulty_data" | "ignore"
        "justification": "",               # at most 150 words for the city's environment office (not graded)
    },
}
registry = []
if client.search_registered_models(filter_string=f"name='{MODEL_NAME}'"):
    for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
        registry.append({"name": MODEL_NAME, "version": int(v.version), "run_id": v.run_id,
                         "aliases": list(client.get_model_version(MODEL_NAME, v.version).aliases)})
with open("answers.json", "w") as f:
    json.dump(answers, f, indent=2)
with open("registry.json", "w") as f:
    json.dump(registry, f, indent=2)
mlflow.search_runs(experiment_names=[EXPERIMENT]).to_csv("mlflow_runs.csv", index=False)
print("wrote answers.json, registry.json and mlflow_runs.csv")

wrote answers.json, registry.json and mlflow_runs.csv


## AI-use statement

TODO: which AI tools you used, for what, and how you checked their output ("None" is valid, but write a full sentence).